# Demonstrating Qiskit Aer Features with a 2x2 Sudoku Puzzle

This notebook demonstrates the various features of Qiskit Aer using a 2x2 Sudoku puzzle with all 4 cells missing. The puzzle is solved using the `ExactCoverQuantumSolver` with pattern encoding.

In [18]:
# Import necessary libraries
from sudoku_nisq import QSudoku
from sudoku_nisq.solvers import ExactCoverQuantumSolver
from qiskit_aer.noise import NoiseModel, depolarizing_error

# Initialize the 4x4 Sudoku puzzle with all cells missing
puzzle = QSudoku.generate(size=2, num_missing_cells=4)
puzzle.set_solver(ExactCoverQuantumSolver, encoding="pattern")

In [19]:
puzzle.draw_circuit()

In [21]:
puzzle.report_resources()

{'puzzle_info': {'hash': '127a87615b3aac7a81a2b53c61e66188ff79c5a3b195ce8604356c21f7cd4087',
  'size': 2,
  'num_missing_cells': 4},
 'solvers': {'ExactCoverQuantumSolver': {'pattern': {'main_circuit': {'depth': 10,
     'gate_counts': {'C24X': 2,
      'C3X': 2,
      'CCX': 96,
      'CX': 96,
      'H': 25,
      'Measure': 4,
      'X': 65},
     'n_gates': 18,
     'n_mcx_gates': 0,
     'n_qubits': 29},
    'backends': {}},
   'simple': {'main_circuit': {'depth': 34,
     'gate_types': {'CircBox': 32, 'H': 9, 'Measure': 8, 'X': 1},
     'n_gates': 50,
     'n_mcx_gates': 0,
     'n_qubits': 45},
    'backends': {}}}}}

In [ ]:
# Run the puzzle on Aer with default settings
result = puzzle.run_aer(shots=1024)
print("Default Aer Simulation:")
print(f"Measured {len(result.get_counts())} unique outcomes")

# Visualize top outcomes
puzzle._solver.counts_plot(result, backend_alias="Aer", show_valid_only=False, top_n=10)

Default Aer Simulation:
Measured 1 unique outcomes


In [ ]:
# Specify simulation method: statevector
result = puzzle.run_aer(
    shots=2048,
    method="statevector",
    optimization_level=2
)
print("Statevector Simulation:")
print(f"Measured {len(result.get_counts())} unique outcomes")

# Visualize and show summary
puzzle._solver.counts_plot(result, backend_alias="Aer", show_valid_only=True, show_summary=True, top_n=10)

Statevector Simulation:
Measured 1 unique outcomes


In [ ]:
# Add noise model and run density matrix simulation
noise = NoiseModel()
noise.add_all_qubit_quantum_error(
    depolarizing_error(0.01, 1), ['u1', 'u2', 'u3', 'h', 's', 't']
})
noise.add_all_qubit_quantum_error(
    depolarizing_error(0.02, 2), ['cx', 'cz', 'swap']
})

result = puzzle.run_aer(
    shots=4096,
    method="density_matrix",
    noise_model=noise
)
print("Density Matrix Simulation with Noise:")
print(f"Measured {len(result.get_counts())} unique outcomes")

# Plot valid-only to see feasible solutions under noise
puzzle._solver.counts_plot(result, backend_alias="Aer (noisy)", show_valid_only=True, show_summary=True)

Density Matrix Simulation with Noise:
Measured 4 unique outcomes


In [9]:
# GPU acceleration example (if available)
from sudoku_nisq.providers import AerProvider

provider = AerProvider()
info = provider.query_available_devices()
if info['has_gpu']:
    result = puzzle.run_aer(
        shots=2048,
        method="statevector",
        device="GPU",
        precision="single"
    )
    print("GPU-Accelerated Simulation:")
    print(f"Measured {len(result.get_counts())} unique outcomes")
else:
    print("GPU not available. Skipping GPU-accelerated simulation.")

GPU not available. Skipping GPU-accelerated simulation.


In [11]:
# Extract top valid bitstring and display selected subsets
counts = result.get_counts()
# Sort by frequency
sorted_counts = sorted(counts.items(), key=lambda kv: kv[1], reverse=True)
best = None
for bitstring, c in sorted_counts:
    bs = bitstring if isinstance(bitstring, str) else ''.join(str(b) for b in bitstring)
    if puzzle._solver._is_valid_solution(bs):
        best = bs
        break
if best is None and sorted_counts:
    # Fallback: take the most frequent outcome
    best = sorted_counts[0][0]
    best = best if isinstance(best, str) else ''.join(str(b) for b in best)
print(f"Top valid bitstring: {best}")

# Decode to subset indices (1s positions)
selected_indices = [i for i, b in enumerate(best) if b == '1']
print(f"Selected subset indices: {selected_indices}")

# Show the corresponding subset entries
subs = puzzle._solver.subsets
selected_subsets = {f'S_{i}': subs.get(f'S_{i}', []) for i in selected_indices}
for k, v in selected_subsets.items():
    print(f"{k}: {v}")

Top valid bitstring: 11
Selected subset indices: [0, 1]
S_0: [(1, 1), ('row', 1, 1), ('col', 1, 1)]
S_1: [(1, 0), ('row', 1, 2), ('col', 0, 2), (0, 1), ('row', 0, 2), ('col', 1, 2)]
